<a href="https://www.kaggle.com/code/robiulhasanjisan/daraz-sentiament-analysis?scriptVersionId=302927878" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/robiulhasanjisan/bangladesh-daraz-reviews-for-sentiment-analysis/Daraz_Master_Reviews_bd.csv


# Bangla E-commerce Review Rating Prediction
In this project, we analyze product reviews from Daraz and build a deep learning model to predict product ratings (1–5 stars) using Natural Language Processing.

The model uses BanglaBERT combined with class imbalance techniques such as Weighted Sampling and Focal Loss.

In [2]:
!pip install --quiet transformers datasets scikit-learn torch

# Import Libraries

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments



# Load and Clean Dataset

We load the dataset containing customer reviews from Daraz.

**Cleaning steps:**

- Keep only relevant columns

- Remove missing values

- Convert ratings to integers

- Filter ratings between 1 and 5

We also inspect the **rating distribution** to understand class imbalance.

In [4]:
data_path = "/kaggle/input/datasets/robiulhasanjisan/bangladesh-daraz-reviews-for-sentiment-analysis/Daraz_Master_Reviews_bd.csv"

df = pd.read_csv(data_path)

df = df[['review_text', 'rating']].dropna()

df['rating'] = df['rating'].round().astype(int)

df = df[df['rating'].between(1,5)]

print("Rating distribution:\n", df['rating'].value_counts())

Rating distribution:
 rating
5    7229
4     602
1     419
3     262
2     123
Name: count, dtype: int64


# Train–Test Split

The dataset is split into:

- 80% training data

- 20% testing data

We use stratified sampling so that each rating class appears proportionally in both datasets.

In [5]:



train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review_text'].tolist(),
    df['rating'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['rating'].tolist()
)


# Tokenization with BanglaBERT


Transformer models cannot process raw text directly.

We must convert text into tokens (numerical representations) using a tokenizer.

We use the tokenizer from BanglaBERT.

**Tokenization steps:**

- Split text into tokens

- Convert tokens into IDs

- Pad sequences to the same length

- Truncate long sequences

In [6]:
MODEL_NAME = "sagorsarker/bangla-bert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

# Custom PyTorch Dataset


We create a custom PyTorch Dataset class to convert tokenized data into tensors.

**Important detail:**

Ratings are converted from 1–5 → 0–4 because neural networks use zero-based indexing.

In [7]:

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx]-1)  # 0-index for 5-class
        return item

# Create Dataset Objects


Now we convert the encoded text into dataset objects for training and testing.

In [8]:
train_dataset = ReviewDataset(train_encodings, train_labels)
test_dataset = ReviewDataset(test_encodings, test_labels)



# Handle Class Imbalance


Customer reviews are usually imbalanced, with many 5-star ratings and few 1-star ratings.

To fix this we use:

**WeightedRandomSampler**

This increases the probability of selecting samples from rare classes during training.

In [9]:
labels_np = np.array(train_labels)

class_counts = np.array([
    len(np.where(labels_np==i)[0]) for i in range(1,6)
])

weights = 1.0 / class_counts

sample_weights = np.array([
    weights[label-1] for label in labels_np
])

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

weights_tensor = torch.tensor(weights, dtype=torch.float)

# Build Custom BERT Model with Focal Loss


We create a custom neural network using:

- BanglaBERT encoder

- Linear classification layer

- Focal Loss

Focal loss focuses more on **difficult and minority examples**, improving performance on rare classes.

In [10]:
class BertWithFocalLoss(nn.Module):

    def __init__(self, model_name, num_labels=5, alpha=None, gamma=2):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

        self.alpha = alpha
        self.gamma = gamma


    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        token_type_ids=None,
        **kwargs
    ):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            **kwargs
        )

        logits = self.classifier(outputs.pooler_output)

        loss = None

        if labels is not None:

            ce_loss = F.cross_entropy(
                logits,
                labels,
                weight=self.alpha.to(labels.device)
            )

            pt = torch.exp(-ce_loss)

            loss = ((1-pt)**self.gamma * ce_loss).mean()

        return {'loss': loss, 'logits': logits}

# Initialize Model


Now we initialize the custom BERT model.

In [11]:
model = BertWithFocalLoss(
    MODEL_NAME,
    alpha=weights_tensor
)

model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sagorsarker/bangla-bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Training Configuration


We define the training settings using the Hugging Face Trainer API.

| Parameter     | Value |
| ------------- | ----- |
| Epochs        | 3     |
| Batch size    | 16    |
| Learning rate | 2e-5  |
| Weight decay  | 0.01  |


In [12]:
training_args = TrainingArguments(

    output_dir="/kaggle/working/bangla_sentiment",

    num_train_epochs=3,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=32,

    logging_steps=50,

    save_steps=500,

    learning_rate=2e-5,

    weight_decay=0.01,

    logging_dir="/kaggle/working/logs",

    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Evaluation Metrics


Two metrics are used:

- Accuracy

- Macro F1 Score

Macro F1 ensures that all rating classes are evaluated equally, even minority classes.

In [13]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average='macro')
    }

# Trainer Initialization


The Trainer API simplifies training and evaluation of transformer models.

In [14]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

#  Apply Weighted Sampling


We override the default training loader to use WeightedRandomSampler, ensuring balanced training.

In [15]:
trainer.get_train_dataloader = lambda: DataLoader(
    train_dataset,
    batch_size=training_args.per_device_train_batch_size,
    sampler=sampler
)

# Train the Model


Now we start training the BanglaBERT model on customer review data.

In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
50,1.432063
100,1.125829
150,0.851283
200,0.723305
250,0.570429
300,0.656008
350,0.435425
400,0.375460
450,0.383126
500,0.262336


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=1296, training_loss=0.35159871883598376, metrics={'train_runtime': 445.5094, 'train_samples_per_second': 46.518, 'train_steps_per_second': 2.909, 'total_flos': 0.0, 'train_loss': 0.35159871883598376, 'epoch': 3.0})

# Model Evaluation


Finally, we evaluate the model on the test dataset and print the classification report.

In [17]:
preds = trainer.predict(test_dataset)

pred_labels = preds.predictions.argmax(-1) + 1

print(classification_report(
    test_labels,
    pred_labels,
    digits=4
))

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


              precision    recall  f1-score   support

           1     0.1229    0.5238    0.1991        84
           2     0.0089    0.0800    0.0160        25
           3     0.0299    0.2308    0.0530        52
           4     0.0628    0.3750    0.1077       120
           5     0.9630    0.0180    0.0353      1446

    accuracy                         0.0747      1727
   macro avg     0.2375    0.2455    0.0822      1727
weighted avg     0.8177    0.0747    0.0485      1727

